# DyT — Dynamic Tanh

源码导航：[core/norm/dyt.py](../../../core/norm/dyt.py) 中的 `DyT`。

Zhu et al. (2025) 在 *Transformers without Normalization* 中提出 **DyT（Dynamic Tanh）**，首次系统性地证明了 Transformer 中的归一化层并非不可替代。DyT 用可学习的双曲正切函数替代 LayerNorm / RMSNorm 的统计计算，完全消除了 reduction 操作，在多个模态（视觉、语言、语音、扩散模型）上取得了与标准归一化相当的性能，同时显著提升了推理速度。

### 1. 理论推导

DyT 的核心洞察来源于对 LayerNorm 输入-输出映射的统计观察：该映射在逐元素层面呈现类 tanh 的 S 型曲线。与其显式计算均值与方差，不如直接用参数化的 tanh 进行有界非线性变换：

$$\text{DyT}(x) = \tanh(\alpha \cdot x) \odot \gamma$$

其中：
- $\alpha > 0$ 为**可学习标量**，控制饱和区的位置；
- $\gamma \in \mathbb{R}^d$ 为**逐通道可学习缩放**（可选）。

**与统计归一化的本质差异：**

| 属性 | LayerNorm | RMSNorm | DyT |
|---|---|---|---|
| 统计量计算 | 均值 + 方差 | RMS | **无** |
| 归一化操作 | 减均值、除标准差 | 除 RMS | **tanh 非线性** |
| 是否有界 | 否（线性变换） | 否（线性变换） | **是（[-γ, γ]）** |
| 跨 token 同步 | 无 | 无 | **无（纯逐元素）** |
| 参数量 | $2d$ | $d$ | $1 + d$（含 γ） |

由于完全逐元素、无 reduction，DyT 避免了 GPU 上的同步开销，特别适合张量并行或序列长度极大的场景。

**训练稳定性注意**：深层网络中 DyT 可能出现子临界信号传播（subcritical signal propagation），对 $\alpha$ 的初始化敏感。默认 $\alpha = 0.5$ 在大多数场景下表现良好；若搭配 Muon 等谱优化器，建议降至 $0.3$ 以下。

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import torch

ROOT = Path.cwd()
while ROOT.name and not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.norm.dyt import DyT

### 2. 形状与 dtype 检查

In [ ]:
torch.manual_seed(0)
x = torch.randn(2, 4, 16) * 3
norm = DyT(16)
y = norm(x)

print("x.shape =", tuple(x.shape))
print("y.shape =", tuple(y.shape))
print("DyT α 初始值:", norm.alpha.item())
assert x.shape == y.shape, "DyT 必须保持输入输出维度一致！"

### 3. 饱和边界验证

tanh 的输出范围天然限制在 $(-1, 1)$，乘以 $\gamma$ 后限制在 $(-\gamma, \gamma)$。对极大输入，DyT 起到软裁剪（soft-clipping）作用。

In [ ]:
norm = DyT(4, use_gamma=True)
with torch.no_grad():
    norm.gamma.fill_(2.0)

x_extreme = torch.randn(10, 4) * 100.0
y = norm(x_extreme)

print("输入最大值:", x_extreme.abs().max().item())
print("输出最大值:", y.abs().max().item())
assert y.abs().max().item() <= 2.0 + 1e-5, "输出应被限制在 [-γ, γ] 内！"

### 4. 小输入下的近似线性区

当 $|\alpha x| \ll 1$ 时，$\tanh(\alpha x) \approx \alpha x$，DyT 近似为线性缩放，不会压缩有效信号。

In [ ]:
norm = DyT(8)
x_small = torch.randn(5, 8) * 0.01
y = norm(x_small)

# 手动计算线性近似：α * γ * x
alpha = norm.alpha.item()
gamma = norm.gamma.detach()
y_approx = torch.tanh(torch.tensor(alpha) * x_small) * gamma

print("DyT 输出最大值:", y.abs().max().item())
print("与手动 tanh 的误差:", (y - y_approx).abs().max().item())
assert y.abs().max().item() < 0.1, "小输入不应进入饱和区！"

### 5. 源码精讲

以下为 `core/norm/dyt.py` 的完整实现：

```python
class DyT(nn.Module):
    def __init__(self, normalized_shape: int, alpha_init: float = 0.5,
                 gamma_init: float = 1.0, use_gamma: bool = True):
        super().__init__()
        # 对数参数化保证 α 始终为正数
        self.log_alpha = nn.Parameter(torch.tensor(math.log(alpha_init),
                                                    dtype=torch.float32))
        if use_gamma:
            self.gamma = nn.Parameter(
                torch.full((normalized_shape,), gamma_init, dtype=torch.float32))
        else:
            self.register_parameter("gamma", None)

    @property
    def alpha(self) -> torch.Tensor:
        return self.log_alpha.exp()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        orig_dtype = x.dtype
        x_fp32 = x.float()
        alpha = self.alpha.to(x_fp32.dtype)
        # 核心：tanh(αx)，替代 mean/var 统计
        out = torch.tanh(alpha * x_fp32)
        if self.gamma is not None:
            out = out * self.gamma.to(x_fp32.dtype)
        return out.to(orig_dtype)
```

关键设计点：
- `log_alpha` 通过对数参数化保证 $\alpha > 0$，避免训练中出现负值导致 tanh 反转。
- `use_gamma=False` 时可退化为纯 $\tanh(\alpha x)$，进一步减少参数量。
- 所有运算均为**逐元素（element-wise）**，不涉及跨 token 或跨通道的 reduction，GPU 核函数效率极高。

---

## 延伸阅读与参考资料

### 核心论文
- **Transformers without Normalization**: Zhu et al., 2025. [arXiv:2503.10622](https://arxiv.org/abs/2503.10622)

### 后续改进与讨论
- **Stronger Normalization-Free Transformers**: Chen et al., 2025. [arXiv:2512.10938](https://arxiv.org/abs/2512.10938) — 提出 Derf（Dynamic erf）
- **Subcritical Signal Propagation at Initialization in Normalization-Free Transformers**: 2025. [arXiv:2604.11890](https://arxiv.org/abs/2604.11890) — 理论分析 DyT 的深层稳定性
- **Does Your Optimizer Care How You Normalize?**: 2025. [arXiv:2604.01563](https://arxiv.org/abs/2604.01563) — DyT/Derf 与 Muon 优化器的耦合问题